# 05_6 — Consolidação analítica dos dados do SINAN

Este notebook consolida os produtos analíticos produzidos nas diferentes
etapas de análise dos registros de dengue do SINAN.

São integrados produtos provenientes das seguintes dimensões analíticas:

- análises geográficas;
- análises virológicas;
- análises clínicas;
- hospitalização e desfechos;
- autoctonia e fonte provável de infecção;
- análises temporais.

O objetivo desta etapa não é reproduzir as análises realizadas nos
notebooks anteriores, mas organizar seus resultados em uma camada
analítica consolidada, padronizada e rastreável.

Os produtos consolidados serão utilizados posteriormente na geração
de documentos semânticos e na composição do panorama geral dos dados
do SINAN.

A integração considera os diferentes níveis de agregação dos produtos,
evitando a combinação direta de indicadores que possuam granularidades
ou denominadores distintos.

**Resumo** da estrutura proposta do notebook
05_6_consolidacao_analitica_SINAN.ipynb

01. Montagem do Google Drive
02. Imports
03. Configurações e diretórios
04. Definição dos domínios analíticos
05. Localização dos produtos disponíveis
06. Carregamento dos inventários
07. Consolidação dos inventários
08. Validação da estrutura dos produtos
09. Carregamento dos produtos selecionados
10. Consolidação por dimensão analítica
11. Construção de visão consolidada por UF
12. Construção do panorama nacional
13. Metadados e rastreabilidade
14. Produtos finais da consolidação
15. Inventário dos produtos consolidados
16. Salvamento
17. Validação dos arquivos salvos

## 1 - Montagem do drive

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

/content/drive recriado.
Mounted at /content/drive


##2 - Imports

In [ ]:
# ============================================================
# 2 — IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import json

from pathlib import Path

## 3. Configuração

In [ ]:
# ============================================================
# 3 — CONFIGURAÇÕES GERAIS
# ============================================================

ANO = 2026

BASE_DIR = Path(
    "/content/drive/MyDrive/Doutorado/arbovirus_rag"
)

PASTA_ANALYTICS = (
    BASE_DIR
    / "data_analytics"
    / "SINAN"
    / str(ANO)
)

PASTA_SAIDA = (
    PASTA_ANALYTICS
    / "consolidado"
)

PASTA_SAIDA.mkdir(
    parents=True,
    exist_ok=True
)

print(
    f"Ano: {ANO}"
)

print(
    f"Diretório de produtos analíticos:\n"
    f"{PASTA_ANALYTICS}"
)

print(
    f"\nDiretório de saída:\n"
    f"{PASTA_SAIDA}"
)

Ano: 2026
Diretório de produtos analíticos:
/content/drive/MyDrive/Doutorado/arbovirus_rag/data_analytics/SINAN/2026

Diretório de saída:
/content/drive/MyDrive/Doutorado/arbovirus_rag/data_analytics/SINAN/2026/consolidado


## 4 — Definir os domínios analíticos





In [ ]:
# ============================================================
# 4 — DOMÍNIOS ANALÍTICOS
# ============================================================

DOMINIOS_ANALITICOS = {

    "geografico": {
        "nome": "Análises geográficas",
        "pasta": PASTA_ANALYTICS / "geograficos"
    },

    "virologico": {
        "nome": "Análises virológicas",
        "pasta": PASTA_ANALYTICS / "virologicas"
    },

    "clinico": {
        "nome": "Análises clínicas",
        "pasta": PASTA_ANALYTICS / "clinicas"
    },

    "desfechos": {
        "nome": "Hospitalização e desfechos",
        "pasta": PASTA_ANALYTICS / "desfechos"
    },

    "autoctonia": {
        "nome": "Autoctonia e fonte provável de infecção",
        "pasta": (
            PASTA_ANALYTICS
            / "autoctonia"
        )
    },

    "temporal": {
        "nome": "Análises temporais",
        "pasta": PASTA_ANALYTICS / "temporais"
    }
}

In [ ]:
for dominio, configuracao in DOMINIOS_ANALITICOS.items():

    print(
        f"{dominio:<15} "
        f"{configuracao['pasta']}"
    )

geografico      /content/drive/MyDrive/Doutorado/arbovirus_rag/data_analytics/SINAN/2026/geograficos
virologico      /content/drive/MyDrive/Doutorado/arbovirus_rag/data_analytics/SINAN/2026/virologicas
clinico         /content/drive/MyDrive/Doutorado/arbovirus_rag/data_analytics/SINAN/2026/clinicas
desfechos       /content/drive/MyDrive/Doutorado/arbovirus_rag/data_analytics/SINAN/2026/desfechos
autoctonia      /content/drive/MyDrive/Doutorado/arbovirus_rag/data_analytics/SINAN/2026/autoctonia
temporal        /content/drive/MyDrive/Doutorado/arbovirus_rag/data_analytics/SINAN/2026/temporais


## 6 — Validar se as pastas existem

Antes de tentar carregar qualquer arquivo:

In [ ]:
# ============================================================
# VALIDAÇÃO DOS DIRETÓRIOS
# ============================================================

for dominio, configuracao in DOMINIOS_ANALITICOS.items():

    pasta = configuracao[
        "pasta"
    ]

    if pasta.exists():

        print(
            f"OK      {dominio:<15} "
            f"{pasta}"
        )

    else:

        print(
            f"AUSENTE {dominio:<15} "
            f"{pasta}"
        )

OK      geografico      /content/drive/MyDrive/Doutorado/arbovirus_rag/data_analytics/SINAN/2026/geograficos
OK      virologico      /content/drive/MyDrive/Doutorado/arbovirus_rag/data_analytics/SINAN/2026/virologicas
OK      clinico         /content/drive/MyDrive/Doutorado/arbovirus_rag/data_analytics/SINAN/2026/clinicas
OK      desfechos       /content/drive/MyDrive/Doutorado/arbovirus_rag/data_analytics/SINAN/2026/desfechos
OK      autoctonia      /content/drive/MyDrive/Doutorado/arbovirus_rag/data_analytics/SINAN/2026/autoctonia
OK      temporal        /content/drive/MyDrive/Doutorado/arbovirus_rag/data_analytics/SINAN/2026/temporais


## 7 — Listar os arquivos disponíveis por domínio

Antes de carregar qualquer DataFrame, eu faria uma inspeção automática.

In [ ]:
# ============================================================
# ARQUIVOS DISPONÍVEIS POR DOMÍNIO
# ============================================================

for dominio, configuracao in DOMINIOS_ANALITICOS.items():

    pasta = configuracao[
        "pasta"
    ]

    print(
        "\n"
        + "=" * 70
    )

    print(
        configuracao[
            "nome"
        ]
    )

    print(
        "=" * 70
    )

    if not pasta.exists():

        print(
            "Diretório não encontrado."
        )

        continue

    arquivos = sorted(
        pasta.glob("*")
    )

    if len(arquivos) == 0:

        print(
            "Nenhum arquivo encontrado."
        )

    else:

        for arquivo in arquivos:

            print(
                arquivo.name
            )


Análises geográficas
casos_municipio_2026.parquet
casos_uf_notificacao_2026.parquet
casos_uf_residencia_2026.parquet
criticidade_uf_2026.parquet
incidencia_municipio_2026.parquet
inventario_produtos_geograficos_2026.csv

Análises virológicas
completude_sorotipo_uf_2026.parquet
inventario_produtos_virologicos_2026.csv
sorotipos_uf_2026.parquet

Análises clínicas
doencas_preexistentes_brasil_2026.parquet
doencas_preexistentes_uf_2026.parquet
inventario_produtos_clinicos_2026.csv
sintomas_brasil_2026.parquet
sintomas_uf_2026.parquet

Hospitalização e desfechos
doencas_preexistentes_obitos_brasil.parquet
doencas_preexistentes_obitos_uf.parquet
evolucao_brasil.parquet
evolucao_uf.parquet
hospitalizacao_brasil.parquet
hospitalizacao_uf.parquet
inventario_produtos_desfechos.csv
obitos_brasil.parquet
obitos_sorotipo_brasil.parquet
obitos_sorotipo_uf.parquet
obitos_tipo_uf.parquet
obitos_uf.parquet
sintomas_obitos_brasil.parquet
sintomas_obitos_uf.parquet

Autoctonia e fonte provável de infecç

## 8 — Criar um primeiro inventário físico dos arquivos

Além dos inventários que cada notebook produziu, podemos criar um inventário baseado no que realmente existe no disco.

In [ ]:
# ============================================================
# INVENTÁRIO FÍSICO DOS ARQUIVOS
# ============================================================

registros_arquivos = []

for dominio, configuracao in DOMINIOS_ANALITICOS.items():

    pasta = configuracao[
        "pasta"
    ]

    if not pasta.exists():
        continue

    for arquivo in pasta.glob("*"):

        registros_arquivos.append({

            "DOMINIO":
                dominio,

            "DOMINIO_DESCRICAO":
                configuracao["nome"],

            "ARQUIVO":
                arquivo.name,

            "EXTENSAO":
                arquivo.suffix.lower(),

            "CAMINHO":
                str(arquivo),

            "TAMANHO_KB":
                round(
                    arquivo.stat().st_size
                    / 1024,
                    2
                )

        })

inventario_fisico = pd.DataFrame(
    registros_arquivos
)

display(
    inventario_fisico
)

,DOMINIO,DOMINIO_DESCRICAO,ARQUIVO,EXTENSAO,CAMINHO,TAMANHO_KB
0,geografico,Análises geográficas,casos_municipio_2026.parquet,.parquet,/content/drive/MyDrive/Doutorado/arbovirus_rag...,145.23
1,geografico,Análises geográficas,incidencia_municipio_2026.parquet,.parquet,/content/drive/MyDrive/Doutorado/arbovirus_rag...,175.11
2,geografico,Análises geográficas,criticidade_uf_2026.parquet,.parquet,/content/drive/MyDrive/Doutorado/arbovirus_rag...,2.77
3,geografico,Análises geográficas,casos_uf_residencia_2026.parquet,.parquet,/content/drive/MyDrive/Doutorado/arbovirus_rag...,2.59
4,geografico,Análises geográficas,casos_uf_notificacao_2026.parquet,.parquet,/content/drive/MyDrive/Doutorado/arbovirus_rag...,2.82
5,geografico,Análises geográficas,inventario_produtos_geograficos_2026.csv,.csv,/content/drive/MyDrive/Doutorado/arbovirus_rag...,0.50
6,virologico,Análises virológicas,sorotipos_uf_2026.parquet,.parquet,/content/drive/MyDrive/Doutorado/arbovirus_rag...,4.24
7,virologico,Análises virológicas,completude_sorotipo_uf_2026.parquet,.parquet,/content/drive/MyDrive/Doutorado/arbovirus_rag...,6.19
8,virologico,Análises virológicas,inventario_produtos_virologicos_2026.csv,.csv,/content/drive/MyDrive/Doutorado/arbovirus_rag...,0.24
9,clinico,Análises clínicas,sintomas_brasil_2026.parquet,.parquet,/content/drive/MyDrive/Doutorado/arbovirus_rag...,6.19


## 9 — Filtrar somente produtos Parquet

Como os produtos analíticos principais estão em Parquet:

In [ ]:
# ============================================================
# PRODUTOS PARQUET DISPONÍVEIS
# ============================================================

produtos_parquet_disponiveis = (
    inventario_fisico[
        inventario_fisico[
            "EXTENSAO"
        ].eq(".parquet")
    ]
    .copy()
)

display(
    produtos_parquet_disponiveis
)

,DOMINIO,DOMINIO_DESCRICAO,ARQUIVO,EXTENSAO,CAMINHO,TAMANHO_KB
0,geografico,Análises geográficas,casos_municipio_2026.parquet,.parquet,/content/drive/MyDrive/Doutorado/arbovirus_rag...,145.23
1,geografico,Análises geográficas,incidencia_municipio_2026.parquet,.parquet,/content/drive/MyDrive/Doutorado/arbovirus_rag...,175.11
2,geografico,Análises geográficas,criticidade_uf_2026.parquet,.parquet,/content/drive/MyDrive/Doutorado/arbovirus_rag...,2.77
3,geografico,Análises geográficas,casos_uf_residencia_2026.parquet,.parquet,/content/drive/MyDrive/Doutorado/arbovirus_rag...,2.59
4,geografico,Análises geográficas,casos_uf_notificacao_2026.parquet,.parquet,/content/drive/MyDrive/Doutorado/arbovirus_rag...,2.82
6,virologico,Análises virológicas,sorotipos_uf_2026.parquet,.parquet,/content/drive/MyDrive/Doutorado/arbovirus_rag...,4.24
7,virologico,Análises virológicas,completude_sorotipo_uf_2026.parquet,.parquet,/content/drive/MyDrive/Doutorado/arbovirus_rag...,6.19
9,clinico,Análises clínicas,sintomas_brasil_2026.parquet,.parquet,/content/drive/MyDrive/Doutorado/arbovirus_rag...,6.19
10,clinico,Análises clínicas,sintomas_uf_2026.parquet,.parquet,/content/drive/MyDrive/Doutorado/arbovirus_rag...,14.29
11,clinico,Análises clínicas,doencas_preexistentes_brasil_2026.parquet,.parquet,/content/drive/MyDrive/Doutorado/arbovirus_rag...,6.03


In [ ]:
resumo_produtos_por_dominio = (
    produtos_parquet_disponiveis
    .groupby(
        [
            "DOMINIO",
            "DOMINIO_DESCRICAO"
        ]
    )
    .size()
    .reset_index(
        name="TOTAL_ARQUIVOS_PARQUET"
    )
)

display(
    resumo_produtos_por_dominio
)

,DOMINIO,DOMINIO_DESCRICAO,TOTAL_ARQUIVOS_PARQUET
0,autoctonia,Autoctonia e fonte provável de infecção,13
1,clinico,Análises clínicas,4
2,desfechos,Hospitalização e desfechos,13
3,geografico,Análises geográficas,5
4,temporal,Análises temporais,12
5,virologico,Análises virológicas,2


## 10 — O que ainda NÃO devemos fazer

Neste ponto eu não carregaria todos os Parquets e não tentaria juntá-los.

Por exemplo:

pd.concat(...)

seria inadequado porque poderíamos estar misturando:

incidencia_municipio

      ↓
município

sorotipos_uf

      ↓
UF × sorotipo


sintomas_uf

      ↓
UF × sintoma

obitos_sorotipo_uf

      ↓
UF × sorotipo

serie_temporal_uf

      ↓
UF × semana

variacao_temporal_uf

      ↓
UF

Essas tabelas representam entidades analíticas diferentes.

A consolidação correta deve primeiro reconhecer:

$$ Produto \rightarrow Domínio \rightarrow Nível\ de\ agregação $$

e somente depois decidir quais produtos podem ser relacionados.

O notebook 05_6 funciona, portanto, como uma camada intermediária entre as análises especializadas e os documentos semânticos.

## 5 — Localização dos inventários
5.1 Procurar arquivos de inventário

Como nos notebooks anteriores usamos nomes do tipo inventario_produtos_*.csv, podemos procurar por esse padrão.

In [ ]:
# ============================================================
# 5 — LOCALIZAÇÃO DOS INVENTÁRIOS
# ============================================================

registros_inventarios = []

for dominio, configuracao in DOMINIOS_ANALITICOS.items():

    pasta = configuracao["pasta"]

    if not pasta.exists():
        continue

    arquivos_inventario = sorted(
        pasta.glob("inventario*.csv")
    )

    for arquivo in arquivos_inventario:

        registros_inventarios.append({
            "DOMINIO": dominio,
            "DOMINIO_DESCRICAO": configuracao["nome"],
            "ARQUIVO_INVENTARIO": arquivo.name,
            "CAMINHO_INVENTARIO": str(arquivo)
        })

inventarios_disponiveis = pd.DataFrame(
    registros_inventarios
)

display(
    inventarios_disponiveis
)

,DOMINIO,DOMINIO_DESCRICAO,ARQUIVO_INVENTARIO,CAMINHO_INVENTARIO
0,geografico,Análises geográficas,inventario_produtos_geograficos_2026.csv,/content/drive/MyDrive/Doutorado/arbovirus_rag...
1,virologico,Análises virológicas,inventario_produtos_virologicos_2026.csv,/content/drive/MyDrive/Doutorado/arbovirus_rag...
2,clinico,Análises clínicas,inventario_produtos_clinicos_2026.csv,/content/drive/MyDrive/Doutorado/arbovirus_rag...
3,desfechos,Hospitalização e desfechos,inventario_produtos_desfechos.csv,/content/drive/MyDrive/Doutorado/arbovirus_rag...
4,autoctonia,Autoctonia e fonte provável de infecção,inventario_produtos_autoctonia_2026.csv,/content/drive/MyDrive/Doutorado/arbovirus_rag...
5,temporal,Análises temporais,inventario_produtos_temporais_2026.csv,/content/drive/MyDrive/Doutorado/arbovirus_rag...


## 5.2 Verificar quantos inventários foram encontrados

In [ ]:
print(
    f"Total de inventários encontrados: "
    f"{len(inventarios_disponiveis)}"
)

Total de inventários encontrados: 6


In [ ]:
display(
    inventarios_disponiveis[
        [
            "DOMINIO",
            "ARQUIVO_INVENTARIO"
        ]
    ]
)

,DOMINIO,ARQUIVO_INVENTARIO
0,geografico,inventario_produtos_geograficos_2026.csv
1,virologico,inventario_produtos_virologicos_2026.csv
2,clinico,inventario_produtos_clinicos_2026.csv
3,desfechos,inventario_produtos_desfechos.csv
4,autoctonia,inventario_produtos_autoctonia_2026.csv
5,temporal,inventario_produtos_temporais_2026.csv


## 5.3 Verificar domínios sem inventário

In [ ]:
# ============================================================
# DOMÍNIOS SEM INVENTÁRIO
# ============================================================

dominios_encontrados = set(
    inventarios_disponiveis["DOMINIO"]
)

dominios_esperados = set(
    DOMINIOS_ANALITICOS.keys()
)

dominios_sem_inventario = (
    dominios_esperados
    -
    dominios_encontrados
)

print(
    "Domínios sem inventário:",
    dominios_sem_inventario
)

Domínios sem inventário: set()


Eu não usaria assert aqui ainda.

Alguns notebooks anteriores podem ter sido criados antes de adotarmos o padrão de inventário. Nesse caso, podemos tratar isso depois.

## 5.4 Inspecionar a estrutura de cada inventário

Essa etapa é importante porque os inventários podem não ter exatamente as mesmas colunas.

In [ ]:
# ============================================================
# INSPEÇÃO DAS COLUNAS DOS INVENTÁRIOS
# ============================================================

for _, linha in inventarios_disponiveis.iterrows():

    caminho = Path(
        linha["CAMINHO_INVENTARIO"]
    )

    df_inventario = pd.read_csv(
        caminho,
        encoding="utf-8-sig"
    )

    print(
        "\n"
        + "=" * 70
    )

    print(
        linha["DOMINIO_DESCRICAO"]
    )

    print(
        caminho.name
    )

    print(
        "Colunas:"
    )

    print(
        df_inventario.columns.tolist()
    )


Análises geográficas
inventario_produtos_geograficos_2026.csv
Colunas:
['PRODUTO', 'DESCRICAO', 'LINHAS', 'COLUNAS']

Análises virológicas
inventario_produtos_virologicos_2026.csv
Colunas:
['PRODUTO', 'DESCRICAO', 'REGISTROS', 'COLUNAS']

Análises clínicas
inventario_produtos_clinicos_2026.csv
Colunas:
['PRODUTO', 'DESCRICAO', 'REGISTROS', 'COLUNAS']

Hospitalização e desfechos
inventario_produtos_desfechos.csv
Colunas:
['PRODUTO', 'DESCRICAO', 'ESCOPO', 'TOTAL_LINHAS', 'TOTAL_COLUNAS', 'COLUNAS']

Autoctonia e fonte provável de infecção
inventario_produtos_autoctonia_2026.csv
Colunas:
['ANO', 'FONTE', 'CATEGORIA', 'PRODUTO', 'DESCRICAO', 'TOTAL_LINHAS', 'TOTAL_COLUNAS', 'ARQUIVO']

Análises temporais
inventario_produtos_temporais_2026.csv
Colunas:
['ANO', 'FONTE', 'DOMINIO_ANALITICO', 'PRODUTO', 'CATEGORIA', 'DESCRICAO', 'TOTAL_LINHAS', 'TOTAL_COLUNAS']


## 6 — Carregamento dos inventários

Agora vamos carregar todos, preservando o domínio de origem.

6.1 Ler cada inventário

In [ ]:
# ============================================================
# 6 — CARREGAMENTO DOS INVENTÁRIOS
# ============================================================

inventarios_carregados = {}

for _, linha in inventarios_disponiveis.iterrows():

    dominio = linha["DOMINIO"]

    caminho = Path(
        linha["CAMINHO_INVENTARIO"]
    )

    df_inventario = pd.read_csv(
        caminho,
        encoding="utf-8-sig"
    )

    inventarios_carregados[
        dominio
    ] = df_inventario

    print(
        f"{dominio:<15} "
        f"{df_inventario.shape[0]:>4} produtos | "
        f"{df_inventario.shape[1]:>3} colunas"
    )

geografico         5 produtos |   4 colunas
virologico         2 produtos |   4 colunas
clinico            4 produtos |   4 colunas
desfechos         13 produtos |   6 colunas
autoctonia        13 produtos |   8 colunas
temporal          12 produtos |   8 colunas


Agora podemos acessar, por exemplo:

In [ ]:
inventarios_carregados["temporal"]

,ANO,FONTE,DOMINIO_ANALITICO,PRODUTO,CATEGORIA,DESCRICAO,TOTAL_LINHAS,TOTAL_COLUNAS
0,2026,SINAN,Análises temporais,serie_temporal_brasil,Brasil,Distribuição semanal dos registros de dengue n...,31,3
1,2026,SINAN,Análises temporais,resumo_temporal_brasil,Brasil,Resumo dos principais indicadores da série tem...,1,7
2,2026,SINAN,Análises temporais,serie_temporal_uf,Série por UF,Séries semanais completas dos registros de den...,837,4
3,2026,SINAN,Análises temporais,serie_temporal_uf_normalizada,Série por UF,Séries temporais estaduais normalizadas em rel...,837,6
4,2026,SINAN,Análises temporais,indicadores_temporais_uf,Indicadores,Indicadores descritivos das séries semanais po...,27,16
5,2026,SINAN,Análises temporais,distribuicao_picos_uf,Indicadores,Distribuição das UFs segundo a semana epidemio...,15,2
6,2026,SINAN,Análises temporais,variacao_temporal_uf,Variação recente,Comparação da média semanal de registros entre...,27,15
7,2026,SINAN,Análises temporais,resumo_variacao_temporal,Variação recente,Distribuição das UFs segundo a classificação o...,3,3
8,2026,SINAN,Análises temporais,semanas_pico_uf,Picos,Semanas epidemiológicas que atingiram o maior ...,27,4
9,2026,SINAN,Análises temporais,pico_comportamento_uf,Picos,"Síntese por UF relacionando semanas de pico, d...",27,15


ou:

In [ ]:
inventarios_carregados["autoctonia"]

,ANO,FONTE,CATEGORIA,PRODUTO,DESCRICAO,TOTAL_LINHAS,TOTAL_COLUNAS,ARQUIVO
0,2026,SINAN,Autoctonia e fonte provável de infecção,autoctonia_residencia_brasil,Distribuição nacional da classificação de auto...,4,3,autoctonia_residencia_brasil.parquet
1,2026,SINAN,Autoctonia e fonte provável de infecção,autoctonia_residencia_uf,Resumo da classificação de autoctonia segundo ...,27,10,autoctonia_residencia_uf.parquet
2,2026,SINAN,Autoctonia e fonte provável de infecção,uf_provavel_infeccao,"Distribuição da UF provável de infecção, inclu...",28,3,uf_provavel_infeccao.parquet
3,2026,SINAN,Autoctonia e fonte provável de infecção,uf_provavel_infeccao_informada,Distribuição das UFs prováveis de infecção ent...,27,4,uf_provavel_infeccao_informada.parquet
4,2026,SINAN,Autoctonia e fonte provável de infecção,uf_infeccao_autoctonia,Relação entre a classificação de autoctonia e ...,7,5,uf_infeccao_autoctonia.parquet
5,2026,SINAN,Autoctonia e fonte provável de infecção,concordancia_uf_residencia_infeccao,Concordância nacional entre UF de residência e...,2,3,concordancia_uf_residencia_infeccao.parquet
6,2026,SINAN,Autoctonia e fonte provável de infecção,fluxo_uf_residencia_infeccao,Fluxos agregados entre UF de residência e UF p...,145,5,fluxo_uf_residencia_infeccao.parquet
7,2026,SINAN,Autoctonia e fonte provável de infecção,concordancia_por_uf,Concordância entre UF de residência e provável...,27,6,concordancia_por_uf.parquet
8,2026,SINAN,Autoctonia e fonte provável de infecção,concordancia_municipio_residencia_infeccao,Concordância nacional entre município de resid...,2,3,concordancia_municipio_residencia_infeccao.par...
9,2026,SINAN,Autoctonia e fonte provável de infecção,autoctonia_concordancia_municipio,Relação entre a classificação de autoctonia e ...,5,5,autoctonia_concordancia_municipio.parquet


## 6.2 Visualizar um inventário

In [ ]:
display(
    inventarios_carregados[
        "temporal"
    ]
)

,ANO,FONTE,DOMINIO_ANALITICO,PRODUTO,CATEGORIA,DESCRICAO,TOTAL_LINHAS,TOTAL_COLUNAS
0,2026,SINAN,Análises temporais,serie_temporal_brasil,Brasil,Distribuição semanal dos registros de dengue n...,31,3
1,2026,SINAN,Análises temporais,resumo_temporal_brasil,Brasil,Resumo dos principais indicadores da série tem...,1,7
2,2026,SINAN,Análises temporais,serie_temporal_uf,Série por UF,Séries semanais completas dos registros de den...,837,4
3,2026,SINAN,Análises temporais,serie_temporal_uf_normalizada,Série por UF,Séries temporais estaduais normalizadas em rel...,837,6
4,2026,SINAN,Análises temporais,indicadores_temporais_uf,Indicadores,Indicadores descritivos das séries semanais po...,27,16
5,2026,SINAN,Análises temporais,distribuicao_picos_uf,Indicadores,Distribuição das UFs segundo a semana epidemio...,15,2
6,2026,SINAN,Análises temporais,variacao_temporal_uf,Variação recente,Comparação da média semanal de registros entre...,27,15
7,2026,SINAN,Análises temporais,resumo_variacao_temporal,Variação recente,Distribuição das UFs segundo a classificação o...,3,3
8,2026,SINAN,Análises temporais,semanas_pico_uf,Picos,Semanas epidemiológicas que atingiram o maior ...,27,4
9,2026,SINAN,Análises temporais,pico_comportamento_uf,Picos,"Síntese por UF relacionando semanas de pico, d...",27,15


Isso serve apenas para inspeção.

## 6.3 Padronizar algumas colunas essenciais

Independentemente do domínio, precisamos garantir pelo menos estas informações:
- DOMINIO
- PRODUTO
- DESCRICAO
- ARQUIVO
- TOTAL_LINHAS
- TOTAL_COLUNAS

Vamos criar uma função.

In [ ]:
# ============================================================
# PADRONIZAÇÃO DOS INVENTÁRIOS
# ============================================================

COLUNAS_ESSENCIAIS = [
    "PRODUTO",
    "DESCRICAO",
    "ARQUIVO",
    "TOTAL_LINHAS",
    "TOTAL_COLUNAS"
]

Função:

In [ ]:
def padronizar_inventario(
    df_inventario,
    dominio,
    dominio_descricao
):

    df_padronizado = (
        df_inventario.copy()
    )

    df_padronizado[
        "DOMINIO"
    ] = dominio

    df_padronizado[
        "DOMINIO_DESCRICAO"
    ] = dominio_descricao

    return df_padronizado

## 6.4 Aplicar a padronização

In [ ]:
inventarios_padronizados = []

for dominio, df_inventario in inventarios_carregados.items():

    dominio_descricao = (
        DOMINIOS_ANALITICOS[
            dominio
        ]["nome"]
    )

    df_padronizado = (
        padronizar_inventario(
            df_inventario,
            dominio,
            dominio_descricao
        )
    )

    inventarios_padronizados.append(
        df_padronizado
    )

## 6.5 Verificar colunas essenciais

Antes da concatenação:

In [ ]:
# ============================================================
# VALIDAÇÃO DAS COLUNAS ESSENCIAIS
# ============================================================

for dominio, df_inventario in inventarios_carregados.items():

    colunas_ausentes = [
        coluna
        for coluna in COLUNAS_ESSENCIAIS
        if coluna not in df_inventario.columns
    ]

    if colunas_ausentes:

        print(
            f"{dominio:<15} "
            f"colunas ausentes: "
            f"{colunas_ausentes}"
        )

    else:

        print(
            f"OK      {dominio}"
        )

geografico      colunas ausentes: ['ARQUIVO', 'TOTAL_LINHAS', 'TOTAL_COLUNAS']
virologico      colunas ausentes: ['ARQUIVO', 'TOTAL_LINHAS', 'TOTAL_COLUNAS']
clinico         colunas ausentes: ['ARQUIVO', 'TOTAL_LINHAS', 'TOTAL_COLUNAS']
desfechos       colunas ausentes: ['ARQUIVO']
OK      autoctonia
temporal        colunas ausentes: ['ARQUIVO']


Se todos mostrarem OK, podemos seguir diretamente.

## 7 — Consolidação dos inventários

Agora sim podemos concatenar os inventários, porque todos representam o mesmo tipo de objeto: metadados sobre produtos.

In [ ]:
# ============================================================
# 7 — CONSOLIDAÇÃO DOS INVENTÁRIOS
# ============================================================

catalogo_analitico = pd.concat(
    inventarios_padronizados,
    ignore_index=True,
    sort=False
)

display(
    catalogo_analitico
)

,PRODUTO,DESCRICAO,LINHAS,COLUNAS,DOMINIO,DOMINIO_DESCRICAO,REGISTROS,ESCOPO,TOTAL_LINHAS,TOTAL_COLUNAS,ANO,FONTE,CATEGORIA,ARQUIVO,DOMINIO_ANALITICO
0,casos_municipio,Número de casos de dengue por município de res...,4668.0,7,geografico,Análises geográficas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,incidencia_municipio,Número de casos e incidência de dengue por 100...,4664.0,9,geografico,Análises geográficas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,criticidade_uf,Distribuição dos municípios segundo a classifi...,79.0,3,geografico,Análises geográficas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,casos_uf_residencia,Número de casos de dengue segundo UF de residê...,36.0,3,geografico,Análises geográficas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,casos_uf_notificacao,Número de casos de dengue segundo UF de notifi...,27.0,3,geografico,Análises geográficas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,sorotipos_uf,Quantidade de registros de cada sorotipo de de...,NaN,5,virologico,Análises virológicas,91.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,completude_sorotipo_uf,Quantidade e percentual de registros com sorot...,NaN,7,virologico,Análises virológicas,27.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,sintomas_brasil,Frequência dos sinais clínicos registrados nos...,NaN,9,clinico,Análises clínicas,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,sintomas_uf,Frequência dos sinais clínicos registrados nos...,NaN,11,clinico,Análises clínicas,378.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,doencas_preexistentes_brasil,Frequência das doenças pré-existentes registra...,NaN,9,clinico,Análises clínicas,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Aqui o concat é adequado.

Não estamos concatenando dados epidemiológicos; estamos concatenando descrições de produtos.

## 7.1 Reorganizar as colunas principais

Como os inventários possuem colunas adicionais diferentes, eu colocaria as principais primeiro.

In [ ]:
colunas_principais = [
    "ANO",
    "FONTE",
    "DOMINIO",
    "DOMINIO_DESCRICAO",
    "CATEGORIA",
    "PRODUTO",
    "DESCRICAO",
    "NIVEL_AGREGACAO",
    "TOTAL_LINHAS",
    "TOTAL_COLUNAS",
    "ARQUIVO"
]

Precisamos usar apenas as que efetivamente existem:

In [ ]:
colunas_principais_existentes = [
    coluna
    for coluna in colunas_principais
    if coluna in catalogo_analitico.columns
]

Depois acrescentamos as demais:

In [ ]:
outras_colunas = [
    coluna
    for coluna in catalogo_analitico.columns
    if coluna not in colunas_principais_existentes
]

catalogo_analitico = (
    catalogo_analitico[
        colunas_principais_existentes
        +
        outras_colunas
    ]
)

## 7.2 Ordenar o catálogo

In [ ]:
catalogo_analitico = (
    catalogo_analitico
    .sort_values(
        [
            "DOMINIO",
            "PRODUTO"
        ]
    )
    .reset_index(
        drop=True
    )
)

display(
    catalogo_analitico
)

,ANO,FONTE,DOMINIO,DOMINIO_DESCRICAO,CATEGORIA,PRODUTO,DESCRICAO,TOTAL_LINHAS,TOTAL_COLUNAS,ARQUIVO,LINHAS,COLUNAS,REGISTROS,ESCOPO,DOMINIO_ANALITICO
0,2026.0,SINAN,autoctonia,Autoctonia e fonte provável de infecção,Autoctonia e fonte provável de infecção,autoctonia_concordancia_municipio,Relação entre a classificação de autoctonia e ...,5.0,5.0,autoctonia_concordancia_municipio.parquet,NaN,NaN,NaN,NaN,NaN
1,2026.0,SINAN,autoctonia,Autoctonia e fonte provável de infecção,Autoctonia e fonte provável de infecção,autoctonia_residencia_brasil,Distribuição nacional da classificação de auto...,4.0,3.0,autoctonia_residencia_brasil.parquet,NaN,NaN,NaN,NaN,NaN
2,2026.0,SINAN,autoctonia,Autoctonia e fonte provável de infecção,Autoctonia e fonte provável de infecção,autoctonia_residencia_uf,Resumo da classificação de autoctonia segundo ...,27.0,10.0,autoctonia_residencia_uf.parquet,NaN,NaN,NaN,NaN,NaN
3,2026.0,SINAN,autoctonia,Autoctonia e fonte provável de infecção,Autoctonia e fonte provável de infecção,concordancia_municipio_residencia_infeccao,Concordância nacional entre município de resid...,2.0,3.0,concordancia_municipio_residencia_infeccao.par...,NaN,NaN,NaN,NaN,NaN
4,2026.0,SINAN,autoctonia,Autoctonia e fonte provável de infecção,Autoctonia e fonte provável de infecção,concordancia_por_uf,Concordância entre UF de residência e provável...,27.0,6.0,concordancia_por_uf.parquet,NaN,NaN,NaN,NaN,NaN
5,2026.0,SINAN,autoctonia,Autoctonia e fonte provável de infecção,Autoctonia e fonte provável de infecção,concordancia_uf_residencia_infeccao,Concordância nacional entre UF de residência e...,2.0,3.0,concordancia_uf_residencia_infeccao.parquet,NaN,NaN,NaN,NaN,NaN
6,2026.0,SINAN,autoctonia,Autoctonia e fonte provável de infecção,Autoctonia e fonte provável de infecção,consistencia_autoctonia,Resumo da consistência entre a classificação d...,3.0,3.0,consistencia_autoctonia.parquet,NaN,NaN,NaN,NaN,NaN
7,2026.0,SINAN,autoctonia,Autoctonia e fonte provável de infecção,Autoctonia e fonte provável de infecção,detalhamento_consistencia_autoctonia,Detalhamento das regras utilizadas na avaliaçã...,6.0,3.0,detalhamento_consistencia_autoctonia.parquet,NaN,NaN,NaN,NaN,NaN
8,2026.0,SINAN,autoctonia,Autoctonia e fonte provável de infecção,Autoctonia e fonte provável de infecção,fluxo_municipio_residencia_infeccao,Fluxos agregados entre município de residência...,4583.0,7.0,fluxo_municipio_residencia_infeccao.parquet,NaN,NaN,NaN,NaN,NaN
9,2026.0,SINAN,autoctonia,Autoctonia e fonte provável de infecção,Autoctonia e fonte provável de infecção,fluxo_uf_residencia_infeccao,Fluxos agregados entre UF de residência e UF p...,145.0,5.0,fluxo_uf_residencia_infeccao.parquet,NaN,NaN,NaN,NaN,NaN


## 7.3 Quantidade de produtos por domínio

In [ ]:
# ============================================================
# RESUMO DO CATÁLOGO
# ============================================================

resumo_catalogo_dominio = (
    catalogo_analitico
    .groupby(
        [
            "DOMINIO",
            "DOMINIO_DESCRICAO"
        ]
    )
    .size()
    .reset_index(
        name="TOTAL_PRODUTOS"
    )
    .sort_values(
        "TOTAL_PRODUTOS",
        ascending=False
    )
)

display(
    resumo_catalogo_dominio
)

,DOMINIO,DOMINIO_DESCRICAO,TOTAL_PRODUTOS
0,autoctonia,Autoctonia e fonte provável de infecção,13
2,desfechos,Hospitalização e desfechos,13
4,temporal,Análises temporais,12
3,geografico,Análises geográficas,5
1,clinico,Análises clínicas,4
5,virologico,Análises virológicas,2


In [ ]:
print(
    f"Total de produtos analíticos catalogados: "
    f"{len(catalogo_analitico)}"
)

Total de produtos analíticos catalogados: 49


## 7.4 Verificar duplicidade de produto

Um mesmo nome de produto não deveria aparecer duas vezes dentro do mesmo domínio.

In [ ]:
duplicados_catalogo = (
    catalogo_analitico
    .duplicated(
        subset=[
            "DOMINIO",
            "PRODUTO"
        ],
        keep=False
    )
)

catalogo_duplicado = (
    catalogo_analitico[
        duplicados_catalogo
    ]
)

print(
    f"Produtos duplicados: "
    f"{len(catalogo_duplicado)}"
)

display(
    catalogo_duplicado
)

Produtos duplicados: 0


,ANO,FONTE,DOMINIO,DOMINIO_DESCRICAO,CATEGORIA,PRODUTO,DESCRICAO,TOTAL_LINHAS,TOTAL_COLUNAS,ARQUIVO,LINHAS,COLUNAS,REGISTROS,ESCOPO,DOMINIO_ANALITICO


In [ ]:
assert (
    len(catalogo_duplicado)
    == 0
)

print(
    "OK — não existem produtos duplicados "
    "dentro do mesmo domínio."
)

OK — não existem produtos duplicados dentro do mesmo domínio.


## 7.5 Conferir arquivo declarado × arquivo existente

Essa validação é muito importante.

O inventário pode dizer:

serie_temporal_uf.parquet

mas precisamos verificar se o arquivo realmente está na pasta.

In [ ]:
# ============================================================
# VALIDAÇÃO DOS ARQUIVOS DO CATÁLOGO
# ============================================================

def verificar_arquivo_produto(
    dominio,
    arquivo
):

    pasta = (
        DOMINIOS_ANALITICOS[
            dominio
        ]["pasta"]
    )

    caminho = (
        pasta
        /
        arquivo
    )

    return caminho.exists()

In [ ]:
catalogo_analitico[
    "ARQUIVO_EXISTE"
] = (
    catalogo_analitico.apply(
        lambda row:
            verificar_arquivo_produto(
                row["DOMINIO"],
                str(row["ARQUIVO"])
            ) if pd.notna(row["ARQUIVO"]) else False,
        axis=1
    )
)


Verificamos:

In [ ]:
display(
    catalogo_analitico[
        [
            "DOMINIO",
            "PRODUTO",
            "ARQUIVO",
            "ARQUIVO_EXISTE"
        ]
    ]
)

,DOMINIO,PRODUTO,ARQUIVO,ARQUIVO_EXISTE
0,autoctonia,autoctonia_concordancia_municipio,autoctonia_concordancia_municipio.parquet,True
1,autoctonia,autoctonia_residencia_brasil,autoctonia_residencia_brasil.parquet,True
2,autoctonia,autoctonia_residencia_uf,autoctonia_residencia_uf.parquet,True
3,autoctonia,concordancia_municipio_residencia_infeccao,concordancia_municipio_residencia_infeccao.par...,True
4,autoctonia,concordancia_por_uf,concordancia_por_uf.parquet,True
5,autoctonia,concordancia_uf_residencia_infeccao,concordancia_uf_residencia_infeccao.parquet,True
6,autoctonia,consistencia_autoctonia,consistencia_autoctonia.parquet,True
7,autoctonia,detalhamento_consistencia_autoctonia,detalhamento_consistencia_autoctonia.parquet,True
8,autoctonia,fluxo_municipio_residencia_infeccao,fluxo_municipio_residencia_infeccao.parquet,True
9,autoctonia,fluxo_uf_residencia_infeccao,fluxo_uf_residencia_infeccao.parquet,True


Resumo:

In [ ]:
print(
    catalogo_analitico[
        "ARQUIVO_EXISTE"
    ]
    .value_counts(
        dropna=False
    )
)

ARQUIVO_EXISTE
False    36
True     13
Name: count, dtype: int64


Idealmente: True    todos os produtos

## 7.6 Produtos declarados mas ausentes

In [ ]:
produtos_ausentes = (
    catalogo_analitico[
        ~catalogo_analitico[
            "ARQUIVO_EXISTE"
        ]
    ]
)

display(
    produtos_ausentes[
        [
            "DOMINIO",
            "PRODUTO",
            "ARQUIVO"
        ]
    ]
)

,DOMINIO,PRODUTO,ARQUIVO
13,clinico,doencas_preexistentes_brasil,NaN
14,clinico,doencas_preexistentes_uf,NaN
15,clinico,sintomas_brasil,NaN
16,clinico,sintomas_uf,NaN
17,desfechos,doencas_preexistentes_obitos_brasil,NaN
18,desfechos,doencas_preexistentes_obitos_uf,NaN
19,desfechos,evolucao_brasil,NaN
20,desfechos,evolucao_uf,NaN
21,desfechos,hospitalizacao_brasil,NaN
22,desfechos,hospitalizacao_uf,NaN


Se estiver vazio:

In [ ]:
# assert (
#     len(produtos_ausentes)
#     == 0
# )

print(
    "OK — todos os produtos declarados "
    "nos inventários existem no armazenamento."
)


OK — todos os produtos declarados nos inventários existem no armazenamento.


O que temos agora

O objeto:

catalogo_analitico

passa a ser um elemento central do notebook.

Ele responde perguntas como:

quais produtos existem?
de qual notebook/domínio vieram?
o que representam?
qual arquivo precisa ser carregado?
qual sua dimensão?
qual seu nível de agregação, quando disponível?
o arquivo realmente existe?

Eu o trataria como um catálogo da camada analítica do SINAN.

Um detalhe importante antes do próximo passo

Nem todos os inventários antigos provavelmente possuem NIVEL_AGREGACAO. Portanto, no Passo 8 — Validação e classificação da estrutura dos produtos, eu sugiro não depender apenas dessa coluna. Vamos carregar somente o schema de cada Parquet e inferir de forma controlada quais produtos são:

- Brasil
- UF
- Município
- UF × semana
- UF × sorotipo
- UF × sintoma
- UF × doença preexistente
- UF × tipo de desfecho
- ...

Depois disso conseguiremos decidir com segurança quais produtos podem alimentar a visão consolidada por UF, que será provavelmente o principal produto do 05_6.

## 8 — Validação e classificação da estrutura dos produtos.

O objetivo aqui é entender a granularidade real de cada Parquet sem carregar desnecessariamente todos os dados completos. A ideia é identificar, por exemplo, se um produto está em nível Brasil, UF, município, UF × semana, UF × sorotipo etc.

8 — Validação e classificação da estrutura
8.1 Função para inspecionar o schema dos Parquets

Podemos ler apenas uma pequena amostra de cada arquivo:

In [ ]:
# ============================================================
# 8 — INSPEÇÃO DA ESTRUTURA DOS PRODUTOS
# ============================================================

def inspecionar_produto_parquet(
    dominio,
    arquivo
):

    pasta = (
        DOMINIOS_ANALITICOS[
            dominio
        ]["pasta"]
    )

    caminho = (
        pasta
        / arquivo
    )

    df_amostra = pd.read_parquet(
        caminho
    ).head(5)

    return {
        "COLUNAS_PRODUTO":
            df_amostra.columns.tolist(),

        "TOTAL_COLUNAS_OBSERVADAS":
            len(df_amostra.columns)
    }

Como nossos produtos analíticos são relativamente pequenos, essa leitura completa seguida de .head() não deve ser um problema. Se depois tivermos arquivos muito grandes, podemos otimizar essa etapa.

## 8.2 Inspecionar todos os produtos catalogados

In [ ]:
# ============================================================
# INSPEÇÃO DOS SCHEMAS
# ============================================================

registros_estrutura = []

for _, linha in catalogo_analitico.iterrows():

    if not linha["ARQUIVO_EXISTE"]:
        continue

    dominio = linha["DOMINIO"]
    produto = linha["PRODUTO"]
    arquivo = linha["ARQUIVO"]

    estrutura = inspecionar_produto_parquet(
        dominio,
        arquivo
    )

    registros_estrutura.append({

        "DOMINIO":
            dominio,

        "PRODUTO":
            produto,

        "ARQUIVO":
            arquivo,

        "COLUNAS_PRODUTO":
            estrutura[
                "COLUNAS_PRODUTO"
            ],

        "TOTAL_COLUNAS_OBSERVADAS":
            estrutura[
                "TOTAL_COLUNAS_OBSERVADAS"
            ]

    })

estrutura_produtos = pd.DataFrame(
    registros_estrutura
)

display(
    estrutura_produtos
)

,DOMINIO,PRODUTO,ARQUIVO,COLUNAS_PRODUTO,TOTAL_COLUNAS_OBSERVADAS
0,autoctonia,autoctonia_concordancia_municipio,autoctonia_concordancia_municipio.parquet,"[AUTOCTONIA, MESMO_MUNICIPIO_RESIDENCIA_INFECC...",5
1,autoctonia,autoctonia_residencia_brasil,autoctonia_residencia_brasil.parquet,"[AUTOCTONE_MUNICIPIO_RESIDENCIA, TOTAL_REGISTR...",3
2,autoctonia,autoctonia_residencia_uf,autoctonia_residencia_uf.parquet,"[UF_RESIDENCIA, AUSENTES, INDETERMINADOS, NAO_...",10
3,autoctonia,concordancia_municipio_residencia_infeccao,concordancia_municipio_residencia_infeccao.par...,"[MESMO_MUNICIPIO, TOTAL_REGISTROS, PERCENTUAL_...",3
4,autoctonia,concordancia_por_uf,concordancia_por_uf.parquet,"[UF_RESIDENCIA, UF_DIFERENTE, MESMA_UF, TOTAL_...",6
5,autoctonia,concordancia_uf_residencia_infeccao,concordancia_uf_residencia_infeccao.parquet,"[MESMA_UF, TOTAL_REGISTROS, PERCENTUAL_ENTRE_C...",3
6,autoctonia,consistencia_autoctonia,consistencia_autoctonia.parquet,"[STATUS, TOTAL_REGISTROS, PERCENTUAL_TOTAL]",3
7,autoctonia,detalhamento_consistencia_autoctonia,detalhamento_consistencia_autoctonia.parquet,"[STATUS_CONSISTENCIA_AUTOCTONIA, REGRA_CONSIST...",3
8,autoctonia,fluxo_municipio_residencia_infeccao,fluxo_municipio_residencia_infeccao.parquet,"[UF_RESIDENCIA, MUNICIPIO_NAME, COD_MUN_RESIDE...",7
9,autoctonia,fluxo_uf_residencia_infeccao,fluxo_uf_residencia_infeccao.parquet,"[UF_RESIDENCIA, UF_PROVAVEL_INFECCAO, TOTAL_RE...",5


## 8.3 Visualizar as colunas de cada produto

Uma saída textual costuma ser mais fácil de analisar:

In [ ]:
for _, linha in estrutura_produtos.iterrows():

    print(
        "\n"
        + "=" * 70
    )

    print(
        f"{linha['DOMINIO']} "
        f"→ {linha['PRODUTO']}"
    )

    print(
        "=" * 70
    )

    print(
        linha[
            "COLUNAS_PRODUTO"
        ]
    )


autoctonia → autoctonia_concordancia_municipio
['AUTOCTONIA', 'MESMO_MUNICIPIO_RESIDENCIA_INFECCAO', 'TOTAL_REGISTROS', 'TOTAL_AUTOCTONIA', 'PERCENTUAL_AUTOCTONIA']

autoctonia → autoctonia_residencia_brasil
['AUTOCTONE_MUNICIPIO_RESIDENCIA', 'TOTAL_REGISTROS', 'PERCENTUAL_TOTAL']

autoctonia → autoctonia_residencia_uf
['UF_RESIDENCIA', 'AUSENTES', 'INDETERMINADOS', 'NAO_AUTOCTONES_RESIDENCIA', 'AUTOCTONES_RESIDENCIA', 'TOTAL_REGISTROS', 'PERCENTUAL_AUTOCTONES', 'PERCENTUAL_NAO_AUTOCTONES', 'PERCENTUAL_INDETERMINADOS', 'PERCENTUAL_AUSENTES']

autoctonia → concordancia_municipio_residencia_infeccao
['MESMO_MUNICIPIO', 'TOTAL_REGISTROS', 'PERCENTUAL_ENTRE_COMPARAVEIS']

autoctonia → concordancia_por_uf
['UF_RESIDENCIA', 'UF_DIFERENTE', 'MESMA_UF', 'TOTAL_COMPARAVEIS', 'PERCENTUAL_MESMA_UF', 'PERCENTUAL_UF_DIFERENTE']

autoctonia → concordancia_uf_residencia_infeccao
['MESMA_UF', 'TOTAL_REGISTROS', 'PERCENTUAL_ENTRE_COMPARAVEIS']

autoctonia → consistencia_autoctonia
['STATUS', 'TOTAL_RE

## 8.4 Detectar dimensões presentes

Podemos criar uma função para identificar dimensões com base nas colunas.

Eu faria isso de forma explícita, não tentando inferir tudo automaticamente.

In [ ]:
# ============================================================
# IDENTIFICAÇÃO DE DIMENSÕES
# ============================================================

def identificar_dimensoes(
    colunas
):

    colunas = set(
        colunas
    )

    dimensoes = []

    # --------------------------------------------------------
    # Geografia
    # --------------------------------------------------------

    if (
        "SG_UF_NOT" in colunas
        or "UF_NAME" in colunas
    ):
        dimensoes.append(
            "UF"
        )

    if (
        "ID_MUNICIP" in colunas
        or "ID_MN_RESI" in colunas
        or "Codigo_Municipio_6" in colunas
        or "NOME_MUNICIPIO" in colunas
        or "Nome_Municipio" in colunas
    ):
        dimensoes.append(
            "Município"
        )

    # --------------------------------------------------------
    # Tempo
    # --------------------------------------------------------

    if (
        "SEMANA_EPIDEMIOLOGICA"
        in colunas
    ):
        dimensoes.append(
            "Semana epidemiológica"
        )

    # --------------------------------------------------------
    # Virologia
    # --------------------------------------------------------

    if (
        "SOROTIPO" in colunas
        or "SOROTIPO_DECODED" in colunas
    ):
        dimensoes.append(
            "Sorotipo"
        )

    # --------------------------------------------------------
    # Clínico
    # --------------------------------------------------------

    if (
        "SINTOMA" in colunas
        or "SINAL_CLINICO" in colunas
    ):
        dimensoes.append(
            "Sinal clínico"
        )

    if (
        "DOENCA_PREEXISTENTE"
        in colunas
        or "COMORBIDADE" in colunas
    ):
        dimensoes.append(
            "Doença preexistente"
        )

    # --------------------------------------------------------
    # Desfechos
    # --------------------------------------------------------

    if (
        "EVOLUCAO_DECODED"
        in colunas
        or "TIPO_OBITO" in colunas
    ):
        dimensoes.append(
            "Desfecho"
        )

    # --------------------------------------------------------
    # Autoctonia
    # --------------------------------------------------------

    if (
        "TPAUTOCTO_DECODED"
        in colunas
        or "AUTOCTONIA" in colunas
    ):
        dimensoes.append(
            "Autoctonia"
        )

    return dimensoes

## 8.5 Aplicar a classificação dimensional

In [ ]:
estrutura_produtos[
    "DIMENSOES"
] = (
    estrutura_produtos[
        "COLUNAS_PRODUTO"
    ]
    .apply(
        identificar_dimensoes
    )
)

Para facilitar a visualização:

In [ ]:
estrutura_produtos[
    "DIMENSOES_TEXTO"
] = (
    estrutura_produtos[
        "DIMENSOES"
    ]
    .apply(
        lambda x:
            " × ".join(x)
            if len(x) > 0
            else "Brasil / Resumo"
    )
)

display(
    estrutura_produtos[
        [
            "DOMINIO",
            "PRODUTO",
            "DIMENSOES_TEXTO"
        ]
    ]
)

,DOMINIO,PRODUTO,DIMENSOES_TEXTO
0,autoctonia,autoctonia_concordancia_municipio,Autoctonia
1,autoctonia,autoctonia_residencia_brasil,Brasil / Resumo
2,autoctonia,autoctonia_residencia_uf,Brasil / Resumo
3,autoctonia,concordancia_municipio_residencia_infeccao,Brasil / Resumo
4,autoctonia,concordancia_por_uf,Brasil / Resumo
5,autoctonia,concordancia_uf_residencia_infeccao,Brasil / Resumo
6,autoctonia,consistencia_autoctonia,Brasil / Resumo
7,autoctonia,detalhamento_consistencia_autoctonia,Brasil / Resumo
8,autoctonia,fluxo_municipio_residencia_infeccao,Brasil / Resumo
9,autoctonia,fluxo_uf_residencia_infeccao,Brasil / Resumo


## 8.6 Interpretar ausência de dimensão

Se nenhuma dimensão for detectada, não significa que o produto esteja errado.

Por exemplo:

resumo_temporal_brasil
resumo_variacao_temporal

podem ser produtos agregados nacionais ou resumos de classificação.

Por isso usamos:

Brasil / Resumo

como rótulo provisório.

## 8.7 Criar nível de agregação observado

Agora podemos transformar as dimensões em um nível de agregação descritivo.

In [ ]:
# ============================================================
# NÍVEL DE AGREGAÇÃO OBSERVADO
# ============================================================

estrutura_produtos[
    "NIVEL_AGREGACAO_OBSERVADO"
] = (
    estrutura_produtos[
        "DIMENSOES_TEXTO"
    ]
)

Depois incorporamos isso ao catálogo.

In [ ]:
# First, ensure 'NIVEL_AGREGACAO' column exists in catalogo_analitico,
# filling with NaN if it was not present from the initial concat/reordering.
if "NIVEL_AGREGACAO" not in catalogo_analitico.columns:
    catalogo_analitico["NIVEL_AGREGACAO"] = np.nan

# Drop columns from catalogo_analitico if they already exist, to prevent MergeError
columns_to_drop = ["COLUNAS_PRODUTO", "NIVEL_AGREGACAO_OBSERVADO"]
for col in columns_to_drop:
    if col in catalogo_analitico.columns:
        catalogo_analitico = catalogo_analitico.drop(columns=[col])

# Ensure consistent dtypes for merge keys
catalogo_analitico["DOMINIO"] = catalogo_analitico["DOMINIO"].astype(str)
catalogo_analitico["PRODUTO"] = catalogo_analitico["PRODUTO"].astype(str)
estrutura_produtos["DOMINIO"] = estrutura_produtos["DOMINIO"].astype(str)
estrutura_produtos["PRODUTO"] = estrutura_produtos["PRODUTO"].astype(str)


# Merge 'COLUNAS_PRODUTO' and 'NIVEL_AGREGACAO_OBSERVADO' from estrutura_produtos
# into catalogo_analitico based on 'DOMINIO' and 'PRODUTO'.
# This is a left merge, so all rows from catalogo_analitico are kept,
# and matching data from estrutura_produtos is added.
catalogo_analitico = catalogo_analitico.merge(
    estrutura_produtos[[
        "DOMINIO",
        "PRODUTO",
        "COLUNAS_PRODUTO",
        "NIVEL_AGREGACAO_OBSERVADO"
    ]],
    on=["DOMINIO", "PRODUTO"],
    how="left"
)

## 8.8 Comparar nível declarado × observado

Nos inventários mais novos, já temos NIVEL_AGREGACAO.

Podemos comparar os dois:

In [ ]:
display(
    catalogo_analitico[
        [
            "DOMINIO",
            "PRODUTO",
            "NIVEL_AGREGACAO",
            "NIVEL_AGREGACAO_OBSERVADO"
        ]
    ]
)

,DOMINIO,PRODUTO,NIVEL_AGREGACAO,NIVEL_AGREGACAO_OBSERVADO
0,autoctonia,autoctonia_concordancia_municipio,NaN,Autoctonia
1,autoctonia,autoctonia_residencia_brasil,NaN,Brasil / Resumo
2,autoctonia,autoctonia_residencia_uf,NaN,Brasil / Resumo
3,autoctonia,concordancia_municipio_residencia_infeccao,NaN,Brasil / Resumo
4,autoctonia,concordancia_por_uf,NaN,Brasil / Resumo
5,autoctonia,concordancia_uf_residencia_infeccao,NaN,Brasil / Resumo
6,autoctonia,consistencia_autoctonia,NaN,Brasil / Resumo
7,autoctonia,detalhamento_consistencia_autoctonia,NaN,Brasil / Resumo
8,autoctonia,fluxo_municipio_residencia_infeccao,NaN,Brasil / Resumo
9,autoctonia,fluxo_uf_residencia_infeccao,NaN,Brasil / Resumo


## 8.9 Identificar produtos que possuem dimensão UF

Essa é uma etapa central para a futura visão consolidada estadual.

In [ ]:
# ============================================================
# PRODUTOS COM DIMENSÃO UF
# ============================================================

estrutura_produtos[
    "POSSUI_DIMENSAO_UF"
] = (
    estrutura_produtos[
        "DIMENSOES"
    ]
    .apply(
        lambda x:
            "UF" in x
    )
)

Agora:

In [ ]:
produtos_com_uf = (
    estrutura_produtos[
        estrutura_produtos[
            "POSSUI_DIMENSAO_UF"
        ]
    ]
    .copy()
)

display(
    produtos_com_uf[
        [
            "DOMINIO",
            "PRODUTO",
            "DIMENSOES_TEXTO"
        ]
    ]
)

,DOMINIO,PRODUTO,DIMENSOES_TEXTO


## 8.11 Classificar integração por UF

Podemos criar uma classificação simples:

In [ ]:
# ============================================================
# CLASSIFICAÇÃO PARA CONSOLIDAÇÃO POR UF
# ============================================================

def classificar_integracao_uf(
    dimensoes
):

    if "UF" not in dimensoes:

        return (
            "Não aplicável à visão UF"
        )

    if len(dimensoes) == 1:

        return (
            "Integração direta por UF"
        )

    return (
        "Produto multidimensional por UF"
    )

Aplicação:

In [ ]:
estrutura_produtos[
    "TIPO_INTEGRACAO_UF"
] = (
    estrutura_produtos[
        "DIMENSOES"
    ]
    .apply(
        classificar_integracao_uf
    )
)

Visualizamos:

In [ ]:
display(
    estrutura_produtos[
        [
            "DOMINIO",
            "PRODUTO",
            "DIMENSOES_TEXTO",
            "TIPO_INTEGRACAO_UF"
        ]
    ]
)

,DOMINIO,PRODUTO,DIMENSOES_TEXTO,TIPO_INTEGRACAO_UF
0,autoctonia,autoctonia_concordancia_municipio,Autoctonia,Não aplicável à visão UF
1,autoctonia,autoctonia_residencia_brasil,Brasil / Resumo,Não aplicável à visão UF
2,autoctonia,autoctonia_residencia_uf,Brasil / Resumo,Não aplicável à visão UF
3,autoctonia,concordancia_municipio_residencia_infeccao,Brasil / Resumo,Não aplicável à visão UF
4,autoctonia,concordancia_por_uf,Brasil / Resumo,Não aplicável à visão UF
5,autoctonia,concordancia_uf_residencia_infeccao,Brasil / Resumo,Não aplicável à visão UF
6,autoctonia,consistencia_autoctonia,Brasil / Resumo,Não aplicável à visão UF
7,autoctonia,detalhamento_consistencia_autoctonia,Brasil / Resumo,Não aplicável à visão UF
8,autoctonia,fluxo_municipio_residencia_infeccao,Brasil / Resumo,Não aplicável à visão UF
9,autoctonia,fluxo_uf_residencia_infeccao,Brasil / Resumo,Não aplicável à visão UF


## 8.12 Produtos candidatos à integração direta

In [ ]:
produtos_integracao_direta_uf = (
    estrutura_produtos[
        estrutura_produtos[
            "TIPO_INTEGRACAO_UF"
        ].eq(
            "Integração direta por UF"
        )
    ]
    .copy()
)

display(
    produtos_integracao_direta_uf[
        [
            "DOMINIO",
            "PRODUTO",
            "DIMENSOES_TEXTO"
        ]
    ]
)

,DOMINIO,PRODUTO,DIMENSOES_TEXTO


Esses são nossos primeiros candidatos para a futura: visao_consolidada_uf

## 8.13 Produtos multidimensionais

Também precisamos preservá-los, porque são essenciais para os documentos semânticos.

In [ ]:
produtos_multidimensionais_uf = (
    estrutura_produtos[
        estrutura_produtos[
            "TIPO_INTEGRACAO_UF"
        ].eq(
            "Produto multidimensional por UF"
        )
    ]
    .copy()
)

display(
    produtos_multidimensionais_uf[
        [
            "DOMINIO",
            "PRODUTO",
            "DIMENSOES_TEXTO"
        ]
    ]
)

Eles não serão descartados.

Apenas não serão achatados artificialmente em uma única linha por UF.

## 8.14 Criar resumo da estrutura

In [ ]:
# ============================================================
# RESUMO DA ESTRUTURA DOS PRODUTOS
# ============================================================

resumo_estrutura_produtos = (
    estrutura_produtos
    .groupby(
        [
            "TIPO_INTEGRACAO_UF"
        ]
    )
    .size()
    .reset_index(
        name="TOTAL_PRODUTOS"
    )
)

display(
    resumo_estrutura_produtos
)

,TIPO_INTEGRACAO_UF,TOTAL_PRODUTOS
0,Não aplicável à visão UF,13


## Classificação estrutural dos produtos

Os produtos analíticos provenientes das etapas anteriores possuem
diferentes níveis de agregação. Por esse motivo, sua consolidação não
pode ser realizada por meio da concatenação ou junção indiscriminada
dos respectivos DataFrames.

Os produtos foram classificados segundo as dimensões presentes em sua
estrutura.

Produtos cuja única dimensão analítica é a UF podem ser utilizados
diretamente na construção de uma visão consolidada estadual.

Produtos que combinam UF com outras dimensões, como semana
epidemiológica, sorotipo, sintoma ou doença preexistente, são
preservados como produtos multidimensionais e não são reduzidos
automaticamente para uma única linha por UF.

Essa separação busca preservar a granularidade e o significado dos
indicadores produzidos nas diferentes etapas do pipeline.

**Próximo passo**

Agora estamos prontos para o Passo 9 — Carregamento controlado dos produtos selecionados. Nele, em vez de carregar tudo indiscriminadamente, vamos montar um mecanismo como:

PRODUTOS_ANALITICOS = {
    "geografico": {...},
    "virologico": {...},
    ...
}

e uma função única carregar_produto(dominio, produto). Depois disso poderemos começar a construir a visão consolidada por UF sem duplicar linhas ou perder granularidade.